In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict
from pathlib import Path

# ----------------------
# Load data
# ----------------------
df_nodes = pd.read_csv(
    '/mnt/archgen/users/xiaowen/Kamenice/1024/IBD/KNC_v12_16cM_fracgp0.7.node.csv',
    names=['Name', 'Phase', 'Degree', 'Sex', 'Yhap'],
    skiprows=1
)

links = pd.read_csv('/mnt/archgen/users/xiaowen/Kamenice/1024/IBD/v12/KNC_v12_16cM_fracgp0.7.csv')  # <-- replace

# ----------------------
# Filter to only EIA/DIA individuals
# ----------------------
valid_phases = {'KNC-EIA', 'KNC-DIA'}
phase_lookup = df_nodes.set_index('Name')['Phase'].to_dict()

links_filtered = links[
    links['iid1'].map(phase_lookup).isin(valid_phases) &
    links['iid2'].map(phase_lookup).isin(valid_phases)
].copy()

# ----------------------
# Count within/cross degrees
# ----------------------
within_counts = defaultdict(int)
cross_counts = defaultdict(int)

for _, row in links_filtered.iterrows():
    iid1, iid2 = row['iid1'], row['iid2']
    phase1 = phase_lookup.get(iid1)
    phase2 = phase_lookup.get(iid2)

    if phase1 == phase2:
        within_counts[iid1] += 1
        within_counts[iid2] += 1
    else:
        cross_counts[iid1] += 1
        cross_counts[iid2] += 1

# ----------------------
# Build new node table
# ----------------------
df_nodes['WithinDegree'] = df_nodes['Name'].map(within_counts).fillna(0).astype(int)
df_nodes['CrossDegree'] = df_nodes['Name'].map(cross_counts).fillna(0).astype(int)
df_nodes['TotalDegree'] = df_nodes['WithinDegree'] + df_nodes['CrossDegree']

# Save updated node table
output_node_path = '/mnt/archgen/users/xiaowen/Kamenice/1024/IBD/KNC.v12.0.EIA_DIA.within_cross.node.csv'
df_nodes.to_csv(output_node_path, index=False)
print(f"New filtered node table saved to: {output_node_path}")

# ----------------------
# Jitter function for overlaps (phase-wide)
# ----------------------
def add_jitter_if_overlap(subset, jitter_amount=0.3):
    jittered_x = []
    jittered_y = []
    for (x, y), group in subset.groupby(['TotalDegree', 'WithinDegree']):
        if len(group) == 1:
            jittered_x.append(x)
            jittered_y.append(y)
        else:
            jittered_x.extend(x + np.random.uniform(-jitter_amount, jitter_amount, size=len(group)))
            jittered_y.extend(y + np.random.uniform(-jitter_amount, jitter_amount, size=len(group)))
    return np.array(jittered_x), np.array(jittered_y)

# ----------------------
# Scatter plots (EIA & DIA only)
# ----------------------
sex_colors = {'M': 'steelblue', 'F': 'salmon'}
phases = ['KNC-EIA', 'KNC-DIA']
jitter_amount = 0.3
outdir = Path('/mnt/archgen/users/xiaowen/Kamenice/1024/IBD/plots')
outdir.mkdir(parents=True, exist_ok=True)

# Axis limits for comparability
x_min, x_max = df_nodes['TotalDegree'].min(), df_nodes['TotalDegree'].max()
y_min, y_max = df_nodes['WithinDegree'].min(), df_nodes['WithinDegree'].max()

for phase in phases:
    subset_phase = df_nodes[df_nodes['Phase'] == phase].copy()

    # Apply jitter detection once per phase
    jitter_x, jitter_y = add_jitter_if_overlap(subset_phase, jitter_amount=jitter_amount)

    # Ensure jittered coordinates are >= 0
    subset_phase['JitterX'] = np.clip(jitter_x, 0, None)
    subset_phase['JitterY'] = np.clip(jitter_y, 0, None)

    plt.figure(figsize=(8, 6), dpi=600, facecolor='white')
    ax = plt.gca()
    ax.set_facecolor('white')

    # Plot reference line y = x behind points
    plt.plot([x_min, x_max], [x_min, x_max],
             color='gray',
             linestyle='--',
             linewidth=1.5,
             zorder=0)  # behind points

    # Plot scatter points for each sex
    for sex, color in sex_colors.items():
        subset_sex = subset_phase[subset_phase['Sex'] == sex]
        plt.scatter(
            subset_sex['JitterX'],
            subset_sex['JitterY'],
            color=color,
            alpha=0.7,
            edgecolor='black',
            s=50,
            label=sex,
            zorder=1  # above reference line
        )
    period = phase[4:]

    plt.xlabel(f'Total IBD Links ({period}-{period} + EIA-DIA)', fontsize=16)
    plt.ylabel(f'Within-phase IBD Links ({period})', fontsize=16)
    #plt.title(f'Total vs. Within-phase IBD Links - {phase}', fontsize=18)
    plt.xlim(x_min - 1, x_max + 1)
    plt.ylim(y_min - 1, y_max + 1)
    plt.legend(title='Sex', fontsize=15, title_fontsize=16)
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()


    # Save as SVG
    svg_path = outdir / f'ibd_total_vs_within_{phase}_EIA_DIA_0226.svg'
    plt.savefig(svg_path, format='svg', bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Saved plot: {svg_path}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict
from pathlib import Path

# ----------------------
# Load data
# ----------------------
df_nodes = pd.read_csv(
    '/mnt/archgen/users/xiaowen/Kamenice/1024/IBD/KNC_v12_16cM_fracgp0.7.node.csv',
    names=['Name', 'Phase', 'Degree', 'Sex', 'Yhap'],
    skiprows=1
)

links = pd.read_csv('/mnt/archgen/users/xiaowen/Kamenice/1024/IBD/v12/KNC_v12_16cM_fracgp0.7.csv')  # <-- replace

# ----------------------
# Filter to only EIA/LBA individuals
# ----------------------
valid_phases = {'KNC-EIA', 'KNC-MLBA'}
phase_lookup = df_nodes.set_index('Name')['Phase'].to_dict()

links_filtered = links[
    links['iid1'].map(phase_lookup).isin(valid_phases) &
    links['iid2'].map(phase_lookup).isin(valid_phases)
].copy()

# ----------------------
# Count within/cross degrees
# ----------------------
within_counts = defaultdict(int)
cross_counts = defaultdict(int)

for _, row in links_filtered.iterrows():
    iid1, iid2 = row['iid1'], row['iid2']
    phase1 = phase_lookup.get(iid1)
    phase2 = phase_lookup.get(iid2)

    if phase1 == phase2:
        within_counts[iid1] += 1
        within_counts[iid2] += 1
    else:
        cross_counts[iid1] += 1
        cross_counts[iid2] += 1

# ----------------------
# Build new node table
# ----------------------
df_nodes['WithinDegree'] = df_nodes['Name'].map(within_counts).fillna(0).astype(int)
df_nodes['CrossDegree'] = df_nodes['Name'].map(cross_counts).fillna(0).astype(int)
df_nodes['TotalDegree'] = df_nodes['WithinDegree'] + df_nodes['CrossDegree']

# Save updated node table
output_node_path = '/mnt/archgen/users/xiaowen/Kamenice/1024/IBD/KNC.v12.0.EIA_LBA.within_cross.node.csv'
df_nodes.to_csv(output_node_path, index=False)
print(f"New filtered node table saved to: {output_node_path}")

# ----------------------
# Jitter function for overlaps
# ----------------------
def add_jitter_if_overlap(subset, jitter_amount=0.15):
    jittered_x = []
    jittered_y = []
    for (x, y), group in subset.groupby(['TotalDegree', 'WithinDegree']):
        if len(group) == 1:
            jittered_x.append(x)
            jittered_y.append(y)
        else:
            jittered_x.extend(x + np.random.uniform(-jitter_amount, jitter_amount, size=len(group)))
            jittered_y.extend(y + np.random.uniform(-jitter_amount, jitter_amount, size=len(group)))
    return np.array(jittered_x), np.array(jittered_y)

# ----------------------
# Scatter plots (EIA & LBA only)
# ----------------------
sex_colors = {'M': 'steelblue', 'F': 'salmon'}
phases = ['KNC-EIA', 'KNC-MLBA']
jitter_amount = 0.3
outdir = Path('/mnt/archgen/users/xiaowen/Kamenice/1024/IBD/plots')
outdir.mkdir(parents=True, exist_ok=True)

# Axis limits for comparability
x_min, x_max = df_nodes['TotalDegree'].min(), df_nodes['TotalDegree'].max()
y_min, y_max = df_nodes['WithinDegree'].min(), df_nodes['WithinDegree'].max()

for phase in phases:
    subset_phase = df_nodes[df_nodes['Phase'] == phase].copy()

    # Apply jitter detection once per phase
    jitter_x, jitter_y = add_jitter_if_overlap(subset_phase, jitter_amount=jitter_amount)

    # Ensure jittered coordinates are >= 0
    subset_phase['JitterX'] = np.clip(jitter_x, 0, None)
    subset_phase['JitterY'] = np.clip(jitter_y, 0, None)

    plt.figure(figsize=(8, 6), dpi=600, facecolor='white')
    ax = plt.gca()
    ax.set_facecolor('white')

    # Plot reference line y = x behind points
    plt.plot([x_min, x_max], [x_min, x_max],
             color='gray',
             linestyle='--',
             linewidth=1.5,
             zorder=0)  # behind points

    # Plot scatter points for each sex
    for sex, color in sex_colors.items():
        subset_sex = subset_phase[subset_phase['Sex'] == sex]
        plt.scatter(
            subset_sex['JitterX'],
            subset_sex['JitterY'],
            color=color,
            alpha=0.7,
            edgecolor='black',
            s=50,
            label=sex,
            zorder=1  # above reference line
        )
    period = phase[4:]
    plt.xlabel(f'Total IBD Links ({period}-{period} + MLBA-EIA)', fontsize=16)
    plt.ylabel(f'Within-phase IBD Links ({period})', fontsize=16)
    #plt.title(f'Total vs. Within-phase IBD Links - {phase}', fontsize=18)
    plt.xlim(x_min - 1, x_max + 1)
    plt.ylim(y_min - 1, y_max + 1)
    plt.legend(title='Sex', fontsize=15, title_fontsize=16)
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()

    # Save as SVG
    svg_path = outdir / f'ibd_total_vs_within_{phase}_EIA_LBA_0226.svg'
    plt.savefig(svg_path, format='svg', bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Saved plot: {svg_path}")


